# CNN vs CNN-Transformer — Kaggle IDS Pipeline

This notebook trains **two** models on the CIC-IDS2017 dataset using identical data splits:

1. **CNN Classifier** — standalone Conv1D + Global Average Pooling
2. **CNN-Transformer** — Conv1D tokenizer + Transformer encoder

After training, a side-by-side accuracy / metric comparison is displayed.

Outputs per model:
- model checkpoint (`.pth`)
- Integrated Gradients ranking CSV
- Grad-CAM ranking CSV (+ optional plot)
- SHAP global importance CSV (+ plots)

In [ ]:
# Cell 1: Setup & install
import os, sys, glob, warnings, importlib
from importlib.metadata import version, PackageNotFoundError

warnings.filterwarnings('ignore')
os.environ['TORCHDYNAMO_DISABLE'] = '1'

REPO_URL = 'https://github.com/samaraweeramethun-eng/CNN-Transformer.git'
REPO_DIR = '/kaggle/working/cnn_transformer_only'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

!pip install -q shap joblib psutil
!pip install -q -e .

# Ensure this repo's package is imported (avoid collisions with similarly named installs)
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale modules from previous runs in this kernel
for name in list(sys.modules):
    if name == 'cnn_transformer_only' or name.startswith('cnn_transformer_only.'):
        del sys.modules[name]

import torch
cnn_transformer_only = importlib.import_module('cnn_transformer_only')

print('module file:', cnn_transformer_only.__file__)
try:
    pkg_ver = version('cnn-transformer-only')
except PackageNotFoundError:
    pkg_ver = getattr(cnn_transformer_only, '__version__', 'unknown')

print('✓ cnn_transformer_only version:', pkg_ver)
print('✓ PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✓ GPU:', torch.cuda.get_device_name(0))

In [ ]:

# Cell 2: Data discovery (auto-detect under /kaggle/input)
import pandas as pd

KAGGLE_DATASET_PATH = ''  # optional: set exact path to a CSV or folder

DATA_PATH = None
PEEK_CSV = None

if KAGGLE_DATASET_PATH and os.path.exists(KAGGLE_DATASET_PATH):
    if os.path.isdir(KAGGLE_DATASET_PATH):
        DATA_PATH = KAGGLE_DATASET_PATH
        folder_csvs = sorted(glob.glob(os.path.join(KAGGLE_DATASET_PATH, '*.csv')))
        if folder_csvs:
            folder_csvs.sort(key=os.path.getsize, reverse=True)
            PEEK_CSV = folder_csvs[0]
    else:
        DATA_PATH = KAGGLE_DATASET_PATH
        PEEK_CSV = KAGGLE_DATASET_PATH

if DATA_PATH is None:
    patterns = [
        '/kaggle/input/**/*TrafficForML*CICFlowMeter*.csv',
        '/kaggle/input/**/cicids*.csv',
        '/kaggle/input/**/*.csv',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    candidates = sorted(set(candidates))

    if candidates:
        candidates.sort(key=os.path.getsize, reverse=True)
        PEEK_CSV = candidates[0]
        candidate_dir = os.path.dirname(PEEK_CSV)
        same_dir_csvs = sorted(glob.glob(os.path.join(candidate_dir, '*.csv')))
        if len(same_dir_csvs) > 1:
            DATA_PATH = candidate_dir
            print('Auto-detected CSV folder:', DATA_PATH)
            print('CSV files in folder:', len(same_dir_csvs))
        else:
            DATA_PATH = PEEK_CSV
            print('Auto-detected CSV:', DATA_PATH)

# Fallback to repo sample if present
if DATA_PATH is None:
    if os.path.exists('data/cicids2017/cicids2017.csv'):
        DATA_PATH = 'data/cicids2017/cicids2017.csv'
    else:
        DATA_PATH = 'data/cicids2017/cicids2017_sample.csv'
    PEEK_CSV = DATA_PATH

# If using /kaggle/input file, symlink into repo data path for convenience
repo_data_dir = 'data/cicids2017'
os.makedirs(repo_data_dir, exist_ok=True)
repo_csv = os.path.join(repo_data_dir, 'cicids2017.csv')
if DATA_PATH.startswith('/kaggle/input') and os.path.isfile(DATA_PATH) and not os.path.exists(repo_csv):
    os.symlink(DATA_PATH, repo_csv)
    DATA_PATH = repo_csv
    PEEK_CSV = repo_csv

if os.path.isdir(DATA_PATH) and PEEK_CSV is None:
    dir_csvs = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))
    if dir_csvs:
        dir_csvs.sort(key=os.path.getsize, reverse=True)
        PEEK_CSV = dir_csvs[0]

print('Using input:', DATA_PATH)
if os.path.isfile(DATA_PATH):
    print('Size (MB):', os.path.getsize(DATA_PATH) / 1024**2)
else:
    print('Input is a folder; training will load all CSV files inside it.')

if PEEK_CSV is None:
    raise FileNotFoundError(f'No CSV file found for preview in: {DATA_PATH}')

df_peek = pd.read_csv(PEEK_CSV, nrows=5)
label_candidates = [c for c in df_peek.columns if 'label' in c.lower()]
print('Preview from:', PEEK_CSV)
print('Columns:', len(df_peek.columns))
label_col = label_candidates[0] if label_candidates else None
print('Label col:', label_col if label_col else 'NOT FOUND')


# ── Helper: collect label counts from a list of CSVs ──────────────────────────
def collect_label_counts(csv_list, label_col):
    """Read CSVs and return {label_string: count} counter."""
    from collections import Counter
    counts = Counter()
    for csv_path in csv_list:
        df = pd.read_csv(csv_path, usecols=[label_col], low_memory=False)
        df[label_col] = df[label_col].astype(str).str.strip()
        counts.update(df[label_col].value_counts().to_dict())
        del df
    return counts


def print_attack_types(label_counts, dataset_name):
    """Print all original attack categories with count & percentage."""
    total = sum(label_counts.values())
    benign_key = next((k for k in label_counts if k.upper() == 'BENIGN'), None)
    n_benign = label_counts.get(benign_key, 0) if benign_key else 0
    attack_types = {k: v for k, v in label_counts.items()
                    if k.upper() != 'BENIGN'}
    n_attack_types = len(attack_types)
    n_attack_rows  = sum(attack_types.values())

    print(f'\n{"=" * 70}')
    print(f'  {dataset_name} — ALL CATEGORIES (original labels before binary conversion)')
    print(f'  Total rows: {total:,}  |  Attack types: {n_attack_types}')
    print(f'{"=" * 70}')
    print(f'  {"#":<4} {"Category":<40} {"Count":>10}  {"Pct":>7}')
    print(f'  {"─"*4} {"─"*40} {"─"*10}  {"─"*7}')

    idx = 1
    # Print BENIGN first
    if benign_key is not None:
        print(f'  {idx:<4} {"BENIGN":<40} {n_benign:>10,}  {100*n_benign/total:>6.2f}%')
        idx += 1

    # Print attack types sorted by count (descending)
    for cat, count in sorted(attack_types.items(), key=lambda x: -x[1]):
        pct = 100.0 * count / total
        print(f'  {idx:<4} {cat:<40} {count:>10,}  {pct:>6.2f}%')
        idx += 1

    # Binary summary
    print(f'\n  {"─"*60}')
    print(f'  BINARY summary:  BENIGN = {n_benign:,} ({100*n_benign/total:.2f}%)  |  '
          f'ATTACK = {n_attack_rows:,} ({100*n_attack_rows/total:.2f}%)')
    print(f'  {"─"*60}')


# ══════════════════════════════════════════════════════════════════════════════
# CIC-IDS2017 — attack type distribution
# ══════════════════════════════════════════════════════════════════════════════
if label_col:
    if os.path.isdir(DATA_PATH):
        csvs_2017 = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))
    else:
        csvs_2017 = [DATA_PATH]
    counts_2017 = collect_label_counts(csvs_2017, label_col)
    print_attack_types(counts_2017, 'CIC-IDS2017')

# ══════════════════════════════════════════════════════════════════════════════
# CIC-IDS2018 — attack type distribution
# ══════════════════════════════════════════════════════════════════════════════
patterns_2018 = [
    '/kaggle/input/**/*2018*.csv',
    '/kaggle/input/**/*CICFlowMeter*.csv',
    '/kaggle/input/**/*.csv',
]
csvs_2018 = []
for p in patterns_2018:
    csvs_2018.extend(glob.glob(p, recursive=True))
# Exclude any 2017 files already used above
csvs_2017_set = set(os.path.abspath(c) for c in (csvs_2017 if label_col else []))
csvs_2018 = sorted(set(
    c for c in csvs_2018
    if '2017' not in os.path.basename(c).lower()
    and os.path.abspath(c) not in csvs_2017_set
))

if csvs_2018:
    hdr_2018 = pd.read_csv(csvs_2018[0], nrows=0)
    hdr_2018.columns = [str(c).strip() for c in hdr_2018.columns]
    lbl_2018 = next((c for c in hdr_2018.columns if 'label' in c.lower()), None)
    if lbl_2018:
        counts_2018 = collect_label_counts(csvs_2018, lbl_2018)
        print_attack_types(counts_2018, 'CIC-IDS2018')
    else:
        print('\nCIC-IDS2018: label column not found.')
else:
    print('\nCIC-IDS2018: no CSV files found under /kaggle/input.')


In [ ]:
# Cell 3: Configure models (shared by CNN classifier & CNN-Transformer)
# Defensive import in case kernel had stale paths before Cell 1 changes
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from cnn_transformer_only.config import CNNTransformerConfig

USE_SAMPLE = ('sample' in DATA_PATH.lower())

# Memory guardrails for Kaggle: increase MAX_ROWS if you have headroom.
MAX_ROWS = 2_000_000 if not USE_SAMPLE else 300_000

cfg = CNNTransformerConfig(
    input_path=DATA_PATH,
    output_dir='/kaggle/working/artifacts',
    csv_chunksize=200_000,
    max_rows=MAX_ROWS,
    epochs=25 if not USE_SAMPLE else 5,
    batch_size=1024 if not USE_SAMPLE else 64,
    val_batch_size=2048 if not USE_SAMPLE else 128,
    lr=1.5e-3,
    num_workers=2,
    d_model=192 if not USE_SAMPLE else 64,
    conv_channels=96 if not USE_SAMPLE else 32,
    num_layers=3 if not USE_SAMPLE else 1,
    num_heads=8 if not USE_SAMPLE else 4,
    d_ff=768 if not USE_SAMPLE else 256,
    cnn_fc_dim=128 if not USE_SAMPLE else 64,
    ig_steps=32 if not USE_SAMPLE else 8,
    ig_samples=512 if not USE_SAMPLE else 128,
    # ── Preprocessing options (NEW) ─────────────────────────────
    grouped_split=True,            # group-based split by source CSV to reduce leakage
    correlation_threshold=0.95,    # drop features with |r| > this (train-based)
    skew_threshold=5.0,            # apply log1p to features with |skew| > this
)

print('Mode:', 'SAMPLE' if USE_SAMPLE else 'FULL')
print('Output dir:', cfg.output_dir)
print('Data cap (rows):', cfg.max_rows if cfg.max_rows > 0 else 'ALL')
print('Config:', cfg.epochs, 'epochs | d_model', cfg.d_model,
      '| conv_channels', cfg.conv_channels, '| cnn_fc_dim', cfg.cnn_fc_dim,
      '| batch', cfg.batch_size)
print('Preprocessing: grouped_split=%s | corr_thresh=%.2f | skew_thresh=%.1f'
      % (cfg.grouped_split, cfg.correlation_threshold, cfg.skew_threshold))

## Part 1 — Standalone CNN Classifier

In [ ]:
# Cell 4: Bootstrap package import path (run once before training)
import sys, os, importlib

SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for name in list(sys.modules):
    if name == 'cnn_transformer_only' or name.startswith('cnn_transformer_only.'):
        del sys.modules[name]

pkg = importlib.import_module('cnn_transformer_only')
print('Using package from:', pkg.__file__)

### Import Bootstrap (Fix for Stale Kernel Modules)
If you see `ModuleNotFoundError` for `cnn_transformer_only.training`, run the next code cell to force imports from this repo before training.

In [ ]:
# Cell 5: Train CNN Classifier
import gc, time, psutil

# Import after bootstrap to avoid stale-module path issues
from cnn_transformer_only.training.cnn_only_trainer import train_cnn_classifier

os.makedirs(cfg.output_dir, exist_ok=True)

ram = psutil.virtual_memory()
print(f'System RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

gc.collect()
t0 = time.time()
cnn_only_path = train_cnn_classifier(cfg)
print('Checkpoint:', cnn_only_path)
print('Elapsed (min):', (time.time()-t0)/60)

In [ ]:
# Cell 6: CNN Classifier — test metrics
cnn_only_ckpt = torch.load(cnn_only_path, map_location='cpu', weights_only=False)
cnn_only_metrics = cnn_only_ckpt.get('test_metrics', {})
print('=== CNN Classifier — Test Set Metrics ===')
for k in ['auc_roc','auc_pr','f1_score','recall','precision','accuracy']:
    print(f'{k}: {cnn_only_metrics.get(k, None)}')
print('Best threshold:', cnn_only_ckpt.get('best_threshold', 0.5))

## Part 2 — CNN-Transformer

In [ ]:
# Cell 8: Train CNN-Transformer
from cnn_transformer_only.training.cnn_trainer import train_cnn_transformer

ram = psutil.virtual_memory()
print(f'System RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

gc.collect()
t0 = time.time()
cnn_transformer_path = train_cnn_transformer(cfg)
print('Checkpoint:', cnn_transformer_path)
print('Elapsed (min):', (time.time()-t0)/60)

In [ ]:
# Cell 9: CNN-Transformer — test metrics
ct_ckpt = torch.load(cnn_transformer_path, map_location='cpu', weights_only=False)
ct_metrics = ct_ckpt.get('test_metrics', {})
print('=== CNN-Transformer — Test Set Metrics ===')
for k in ['auc_roc','auc_pr','f1_score','recall','precision','accuracy']:
    print(f'{k}: {ct_metrics.get(k, None)}')
print('Best threshold:', ct_ckpt.get('best_threshold', 0.5))

In [ ]:
# Cell 10: View IG + Grad-CAM top features (both models)
import pandas as pd
OUT = cfg.output_dir

for prefix, label in [('cnn_classifier', 'CNN Classifier'), ('cnn_transformer', 'CNN-Transformer')]:
    print(f'\n{"="*50}')
    print(f'{label} — Interpretability')
    print(f'{"="*50}')

    ig_path = os.path.join(OUT, f'{prefix}_integrated_gradients.csv')
    cam_path = os.path.join(OUT, f'{prefix}_grad_cam.csv')

    if os.path.exists(ig_path):
        print(f'=== Top 15 — Integrated Gradients ({label}) ===')
        display(pd.read_csv(ig_path).head(15))
    else:
        print('Missing:', ig_path)

    if os.path.exists(cam_path):
        print(f'=== Top 15 — Grad-CAM ({label}) ===')
        display(pd.read_csv(cam_path).head(15))
    else:
        print('Missing:', cam_path)

In [ ]:
# Cell 11: Run SHAP for both models (optional, can be slow)
from cnn_transformer_only.interpretability.shap_runner import run_shap

shap_root = os.path.join(cfg.output_dir, 'shap')
os.makedirs(shap_root, exist_ok=True)

SHAP_BG = 2000 if not USE_SAMPLE else 200
SHAP_EVAL = 2000 if not USE_SAMPLE else 200
SHAP_POOL = 150_000 if not USE_SAMPLE else 500

shap_outputs = {}
for label, ckpt_path, subdir in [
    ('CNN Classifier', cnn_only_path, 'cnn_classifier'),
    ('CNN-Transformer', cnn_transformer_path, 'cnn_transformer'),
]:
    out_dir = os.path.join(shap_root, subdir)
    os.makedirs(out_dir, exist_ok=True)

    print(f'\nRunning SHAP for: {label}')
    csv_path = run_shap(
        checkpoint_path=ckpt_path,
        data_path=DATA_PATH,
        output_dir=out_dir,
        background_size=SHAP_BG,
        eval_size=SHAP_EVAL,
        eval_pool=SHAP_POOL,
        chunk_size=256,
    )
    shap_outputs[label] = csv_path

    print('SHAP CSV:', csv_path)
    if os.path.exists(csv_path):
        display(pd.read_csv(csv_path).head(15))

## Part 3 — Model Comparison

In [ ]:
# Cell 12: Compare explainability across models (IG / Grad-CAM / SHAP)
import os
import pandas as pd

OUT = cfg.output_dir
EXPLAIN_TOP_K = 20


def top_features(csv_path, value_col, top_k=20):
    if not os.path.exists(csv_path):
        return []
    df = pd.read_csv(csv_path)
    if 'feature' not in df.columns or value_col not in df.columns:
        return []
    return df.sort_values(value_col, ascending=False)['feature'].head(top_k).tolist()


def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def overlap_count(a, b):
    return len(set(a) & set(b))


expl_files = {
    'CNN Classifier': {
        'IG': (os.path.join(OUT, 'cnn_classifier_integrated_gradients.csv'), 'avg_abs_integrated_grad'),
        'Grad-CAM': (os.path.join(OUT, 'cnn_classifier_grad_cam.csv'), 'grad_cam_importance'),
        'SHAP': (os.path.join(OUT, 'shap', 'cnn_classifier', 'shap_global_importance_attack.csv'), 'mean_abs_shap'),
    },
    'CNN-Transformer': {
        'IG': (os.path.join(OUT, 'cnn_transformer_integrated_gradients.csv'), 'avg_abs_integrated_grad'),
        'Grad-CAM': (os.path.join(OUT, 'cnn_transformer_grad_cam.csv'), 'grad_cam_importance'),
        'SHAP': (os.path.join(OUT, 'shap', 'cnn_transformer', 'shap_global_importance_attack.csv'), 'mean_abs_shap'),
    },
}

rows = []
for method in ['IG', 'Grad-CAM', 'SHAP']:
    cnn_top = top_features(*expl_files['CNN Classifier'][method], top_k=EXPLAIN_TOP_K)
    ct_top = top_features(*expl_files['CNN-Transformer'][method], top_k=EXPLAIN_TOP_K)

    rows.append({
        'Method': method,
        'Top-K': EXPLAIN_TOP_K,
        'Overlap_Count': overlap_count(cnn_top, ct_top),
        'Jaccard': jaccard(cnn_top, ct_top),
        'CNN_Only_Top5': ', '.join(cnn_top[:5]),
        'CNN_Transformer_Top5': ', '.join(ct_top[:5]),
    })

explain_cmp = pd.DataFrame(rows)
print('=' * 70)
print('EXPLAINABILITY COMPARISON (CNN Classifier vs CNN-Transformer)')
print('=' * 70)
display(explain_cmp)

# Optional: aggregate feature consensus inside each model across IG/Grad-CAM/SHAP
for model_name in ['CNN Classifier', 'CNN-Transformer']:
    method_tops = [
        top_features(*expl_files[model_name]['IG'], top_k=EXPLAIN_TOP_K),
        top_features(*expl_files[model_name]['Grad-CAM'], top_k=EXPLAIN_TOP_K),
        top_features(*expl_files[model_name]['SHAP'], top_k=EXPLAIN_TOP_K),
    ]

    consensus = set(method_tops[0]) & set(method_tops[1]) & set(method_tops[2])
    print(f'\n{model_name} consensus features across IG, Grad-CAM, SHAP ({len(consensus)}):')
    if consensus:
        print(', '.join(sorted(consensus)[:30]))
    else:
        print('No common features across all three methods.')

In [ ]:
# Cell 13: Side-by-side comparison of CNN Classifier vs CNN-Transformer
import pandas as pd
import matplotlib.pyplot as plt

metric_keys = ['accuracy', 'auc_roc', 'auc_pr', 'f1_score', 'precision', 'recall']

comparison = pd.DataFrame({
    'Metric': metric_keys,
    'CNN Classifier': [cnn_only_metrics.get(k, None) for k in metric_keys],
    'CNN-Transformer': [ct_metrics.get(k, None) for k in metric_keys],
}).set_index('Metric')

comparison['Δ (Transformer − CNN)'] = comparison['CNN-Transformer'] - comparison['CNN Classifier']

print('=' * 65)
print('MODEL COMPARISON — Test Set')
print('=' * 65)
display(comparison)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
comparison[['CNN Classifier', 'CNN-Transformer']].plot.bar(ax=ax, rot=25)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('CNN Classifier vs CNN-Transformer — Test Metrics')
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'model_comparison.png'), dpi=160, bbox_inches='tight')
plt.show()

# Print winner per metric
print('\nPer-metric winner:')
for metric in metric_keys:
    cnn_val = cnn_only_metrics.get(metric, 0)
    ct_val = ct_metrics.get(metric, 0)
    winner = 'CNN-Transformer' if ct_val > cnn_val else ('CNN Classifier' if cnn_val > ct_val else 'Tie')
    print(f'  {metric:12s}: {winner} ({ct_val:.4f} vs {cnn_val:.4f})')